In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 20.9 MB/s  0:00:00m0:00:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2


## Local Inference on GPU 
Model page: https://huggingface.co/ProsusAI/finbert

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ProsusAI/finbert)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="ProsusAI/finbert")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [2]:
# pip install -U transformers accelerate
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert", device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
text = "The company reported strong revenue growth."

result = pipe(text, top_k=None)
print(result)

[{'label': 'positive', 'score': 0.9531024098396301}, {'label': 'neutral', 'score': 0.03277944400906563}, {'label': 'negative', 'score': 0.014118149876594543}]


In [4]:
import torch

text = "The company reported strong revenue growth."

inputs = tokenizer(text, return_tensors="pt")

device = next(model.parameters()).device
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probabilities = torch.softmax(logits, dim=-1)

print("Logits:", logits)
print("Probabilities:", probabilities)

Logits: tensor([[ 2.1083, -2.1040, -1.2617]], device='cuda:0')
Probabilities: tensor([[0.9531, 0.0141, 0.0328]], device='cuda:0')


## Testing news text

In [5]:
test_texts = [
    "The company reported record revenue and strong earnings.",
    "The firm announced significant layoffs after disappointing results.",
    "The company maintained its earnings guidance.",
    "Shares declined following weaker-than-expected quarterly results.",
    "The company reported revenue in line with analyst expectations.",
    "Nothing new to the company.",
    "Massive downfall to the Tesla after the war striking USA.",
    "Political observers are watching the Texas gubernatorial election for signs of an artificial intelligence data center backlash in the 2026 U.S. midterm elections on Tuesday, and prediction market traders see growing odds of the Democratic candidate pulling off an upset."
]

results = pipe(test_texts, top_k=None)

for text, result in zip(test_texts, results):
    print("\nText:", text)
    print(result)


Text: The company reported record revenue and strong earnings.
[{'label': 'positive', 'score': 0.9470416307449341}, {'label': 'neutral', 'score': 0.039386894553899765}, {'label': 'negative', 'score': 0.013571476563811302}]

Text: The firm announced significant layoffs after disappointing results.
[{'label': 'negative', 'score': 0.9727678298950195}, {'label': 'neutral', 'score': 0.0194539912045002}, {'label': 'positive', 'score': 0.007778194732964039}]

Text: The company maintained its earnings guidance.
[{'label': 'positive', 'score': 0.6881186962127686}, {'label': 'neutral', 'score': 0.2923906147480011}, {'label': 'negative', 'score': 0.019490676000714302}]

Text: Shares declined following weaker-than-expected quarterly results.
[{'label': 'negative', 'score': 0.9695258140563965}, {'label': 'neutral', 'score': 0.02303312160074711}, {'label': 'positive', 'score': 0.007441041059792042}]

Text: The company reported revenue in line with analyst expectations.
[{'label': 'positive', 'score

## Remote Inference via Inference Providers 
Ensure you have a valid **HF_TOKEN** set in your environment. You can get your token from [your settings page](https://huggingface.co/settings/tokens). Note: running this may incur charges above the free tier.
The following Python example shows how to run the model remotely on HF Inference Providers, automatically selecting an available inference provider for you. 
For more information on how to use the Inference Providers, please refer to our [documentation and guides](https://huggingface.co/docs/inference-providers/en/index).

In [17]:
# import os
# os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [18]:
# import os
# from huggingface_hub import InferenceClient

# client = InferenceClient(
#     provider="auto",
#     api_key=os.environ["HF_TOKEN"],
# )

# result = client.text_classification(
#     "I like you. I love you",
#     model="ProsusAI/finbert",
# )